In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from analysis.loaders import Session

pd.set_option("display.float_format", lambda v: f"{v:.3f}")

In [ ]:
from pathlib import Path

# A session dir OR the .h5 path both work.
data_dir = Path("/Users/hakan/Library/CloudStorage/Dropbox/Hakan/lab/data/cheese_data/HK1/HK1_20260810/HK1_20260810_005/")
s = Session(data_dir)
s

In [ ]:
# Photodiode display-sync check. These are all built-in Session.trials columns:
#   latency_ms          = (true_onset_t - stim_onset_t) * 1e3   photodiode-measured display latency
#   first_pulse_lat_ms  = (first sync pulse - stim_onset_t) * 1e3
#   sync_ok             = 1 if the online onset-sync landed that trial
t = s.trials
print(f"{s.n_trials} trials | sync_ok {int(t['sync_ok'].sum())}/{len(t)} | "
      f"display latency median {t['latency_ms'].median():.1f} ms "
      f"[{t['latency_ms'].min():.1f}, {t['latency_ms'].max():.1f}]")
t[["stim_az_deg", "sync_ok", "latency_ms", "first_pulse_lat_ms"]]

In [ ]:
# Corrected stim onset (true_onset_t) + reward delivery — also built-in Session.trials columns:
#   reward_lat_ms = (reward_t - true_onset_t) * 1e3        reward timing vs the corrected onset
#   rt_ms         = (first_lick_t - response_window_t)*1e3  first in-window lick vs window open
#   pavlovian     = True ONLY for the L2 Pavlovian-delay rescue reward; operant (lick-triggered)
#                   AND L1 free rewards are both False (the flag marks the rescue, not "auto")
cols = ["stim_az_deg", "trial_outcome", "sync_ok", "latency_ms",
        "reward_ul", "pavlovian", "reward_lat_ms", "rt_ms"]
s.trials[cols]

In [ ]:
# Raw per-trial sync pulses live in Session.pulses (one row per rising edge, t_rel_onset_ms =
# pulse time - stim_onset_t). The first pulse of each trial is the display-onset marker used
# for latency above; count + first pulse per trial:
s.pulses.groupby("trial_num")["t_rel_onset_ms"].agg(n_pulses="count", first_pulse_ms="first")